# T Industries - Genie Agents (idempotent)
Creates or updates two Genie Agents through the Genie REST API, then links **T Sales & Ops** to the *T Commercial Overview* dashboard:

| Genie Agent | Data | Typical questions |
| --- | --- | --- |
| **T Sales & Ops** | metric view `t_sales_metrics`, `gold_price_benchmark`, `gold_product_margin`, `gold_inventory_position` (row-filtered), `gold_ticket_insights`, dims (all in schema `t_hardware`) | Which SKUs are priced >10% above competitors? Top customers by margin in BC last quarter. What's our equivalent of this product? (the photo is described by Genie One, the agent matches the description to the catalog) |
| **T Docs & SOPs** | raw SharePoint files in volume `t_hardware.knowledge` (PDF / DOCX / Markdown, read in Agent mode) + `gold_ticket_insights`, `gold_dim_product` | What is the restocking fee? What's the warranty on commercial closers, and how many warranty claims did we get on closers? |

Every example SQL is executed first, so a broken example fails the run instead of silently teaching Genie bad SQL.

In [0]:
import json, uuid
from databricks.sdk import WorkspaceClient
dbutils.widgets.text("catalog", "mdk_demo_catalog")
dbutils.widgets.text("warehouse_id", "1f421abc4c14022f")
dbutils.widgets.text("dashboard_name", "T Commercial Overview")
C = dbutils.widgets.get("catalog"); WH = dbutils.widgets.get("warehouse_id")
G = S = f"{C}.t_hardware"     # single schema; table names carry the layer prefix (gold_, silver_)
w = WorkspaceClient(); api = w.api_client.do
ME = w.current_user.me().user_name

def ids(n):
    """Genie returns list items by ascending id - generate sorted ids so items keep the order we write them in."""
    return sorted(uuid.uuid4().hex for _ in range(n))

def check(sql):
    spark.sql(sql).limit(1).collect()     # fail fast on a broken example
    return sql

def space_json(tables, instructions, examples, samples, functions=(), benchmarks=(), volumes=()):
    """volumes: (path, description) pairs - raw files the agent reads in Agent mode (indexing takes a few minutes after attach)."""
    ex_ids, s_ids, f_ids, b_ids = ids(len(examples)), ids(len(samples)), ids(len(functions)), ids(len(benchmarks))
    data_sources = {"tables": [{"identifier": t} for t in sorted(tables)]}
    if volumes: data_sources["volumes"] = [{"path": p, "description": [d]} for p, d in volumes]
    return json.dumps({
        "version": 2,
        "config": {"sample_questions": [{"id": i, "question": [q]} for i, q in zip(s_ids, samples)]},
        "data_sources": data_sources,
        "instructions": {
            "text_instructions": [{"id": uuid.uuid4().hex, "content": [l + "\n" for l in instructions]}],
            "example_question_sqls": [{"id": i, "question": [q], "sql": [check(s)]} for i, (q, s) in zip(ex_ids, examples)],
            "sql_functions": [{"id": i, "identifier": f} for i, f in zip(f_ids, functions)],
        },
        "benchmarks": {"questions": [{"id": i, "question": [q], "answer": [{"format": "SQL", "content": [check(s)]}]} for i, (q, s) in zip(b_ids, benchmarks)]},
    })

def upsert_space(title, description, serialized):
    existing = [s for s in api("GET", "/api/2.0/genie/spaces").get("spaces", []) if s["title"] == title]
    body = {"title": title, "description": description, "warehouse_id": WH, "serialized_space": serialized}
    if existing:
        sid = existing[0]["space_id"]; api("PATCH", f"/api/2.0/genie/spaces/{sid}", body=body); print("updated", title, sid)
    else:
        sid = api("POST", "/api/2.0/genie/spaces", body={**body, "parent_path": f"/Users/{ME}"})["space_id"]; print("created", title, sid)
    return sid

In [0]:
# Closest T equivalent of a product described in text (Genie One / TAI describes the customer's photo; this agent never sees it).
# Rank: exact finish, then same finish tone (silver / dark / brass), then trailing-12-month net sales.
TONE = "CASE WHEN {f} IN ('Satin Chrome','Satin Stainless','Polished Chrome') THEN 'silver' WHEN {f} IN ('Matte Black','Oil Rubbed Bronze') THEN 'dark' ELSE 'brass' END"
def equivalent_sql(subcategory, finish):
    return f"""WITH target AS (SELECT '{subcategory}' AS subcategory, '{finish}' AS finish),
c AS (
  SELECT p.sku, p.product_name, p.subcategory, p.finish, p.ansi_grade,
    b.t_list_price_cad, b.competitor_min_price_cad, b.cheapest_competitor, b.competitor_avg_price_cad, b.pct_gap_vs_min, b.price_position,
    CASE WHEN p.finish = t.finish THEN 'exact finish'
         WHEN ({TONE.format(f='p.finish')}) = ({TONE.format(f='t.finish')}) THEN 'similar finish'
         ELSE 'different finish' END AS finish_match,
    m.net_sales_t12m_cad
  FROM {G}.gold_dim_product p
  CROSS JOIN target t
  JOIN {G}.gold_price_benchmark b ON b.sku = p.sku
  LEFT JOIN {G}.gold_product_margin m ON m.sku = p.sku
  WHERE p.subcategory = t.subcategory AND p.is_active)
SELECT row_number() OVER (ORDER BY CASE finish_match WHEN 'exact finish' THEN 0 WHEN 'similar finish' THEN 1 ELSE 2 END, net_sales_t12m_cad DESC NULLS LAST) AS match_rank,
  sku, product_name, subcategory, finish, finish_match, ansi_grade, t_list_price_cad, competitor_min_price_cad, cheapest_competitor, competitor_avg_price_cad, pct_gap_vs_min, price_position
FROM c ORDER BY match_rank LIMIT 3"""

sales_instructions = [
    "You answer sales, margin, pricing, inventory and service questions for T Industries, a Canadian distributor of door hardware (locks, levers, door closers, hinges, exit devices) and bathroom fixtures (faucets, grab bars, bath accessories). All money is CAD.",
    f"Use the metric view {G}.t_sales_metrics with MEASURE() for net sales, gross margin, gross margin %, units, ASP, backorder rate and customer counts. Never re-derive these from gold_sales_daily.",
    "Net sales = gross sales - discounts - returns, excluding freight and tax; cancelled orders are excluded (Finance rule). Backorder rate = backordered units / ordered units (Operations rule; target < 5%, > 10% for a category triggers a supply review).",
    "Regions are BC, ON and US (customer region). Warehouses are 'Delta BC' and 'Toronto'. 'Last quarter' means the previous complete calendar quarter: order_quarter = add_months(date_trunc('QUARTER', current_date()), -3).",
    f"Competitor pricing: {G}.gold_price_benchmark has one row per SKU. pct_gap_vs_min > 0.10 means T's list price is more than 10% above the cheapest competitor. Use {G}.gold_price_benchmark_by_competitor for per-competitor detail (NorthBuild Supply, Pacific Hardware Co, DoorPro Direct).",
    "Product equivalents from a description: customers often send a photo of another brand's product (e.g. Taymor, Schlage). The photo is described by the calling assistant (Genie One / TAI), so you receive a text description such as product type, lock function, finish and grade. You never see the image - do not claim to. Find our closest equivalent the way the example 'closest equivalent of a described product' does: map the description to one subcategory in gold_dim_product, map the finish, then rank active SKUs by finish closeness (exact, then same tone, then other) and trailing-12-month net sales, and return the top 3 with their competitor price benchmark.",
    "Subcategory mapping: keypad, digital or smart lock -> Electronic Keypad Lock; keyed deadbolt -> Deadbolt; keyed entry knob or lever -> Cylindrical Lever Lock; passage or hall/closet knob or lever -> Passage Lever Set; privacy (bed/bath) knob or lever -> Privacy Lever Set; storeroom or classroom lever -> Storeroom Lever Set; commercial mortise lock with lever trim -> Grade 1 Mortise Lock; exterior handleset -> Commercial Entry Lever for the handle, with a Deadbolt as a companion item. We sell levers but no knobs or handlesets - say so when the photo shows one.",
    "Finish mapping: satin nickel / brushed nickel -> Satin Stainless; brushed or satin chrome -> Satin Chrome; chrome or bright chrome -> Polished Chrome; black -> Matte Black; bronze or dark bronze -> Oil Rubbed Bronze; brass or gold -> Satin Brass. Finish tones: silver (Satin Chrome, Satin Stainless, Polished Chrome), dark (Matte Black, Oil Rubbed Bronze), brass (Satin Brass).",
    "In the answer, recommend rank 1 with its list price, cheapest competitor price and gap; list the two alternatives; state the subcategory and finish you mapped to, and say when the finish or product type is not an exact match in our catalog.",
    f"{G}.gold_inventory_position is row-filtered: users only see the warehouses they are entitled to. If a warehouse is missing from results, say the user may not have access to it.",
    "Show percentages as percentages with one decimal. Round currency to whole dollars unless asked for unit prices.",
]
sales_examples = [
    ("Which SKUs are priced more than 10% above competitors?",
     f"SELECT sku, product_name, category, t_list_price_cad, competitor_min_price_cad, cheapest_competitor, competitor_avg_price_cad, pct_gap_vs_min FROM {G}.gold_price_benchmark WHERE pct_gap_vs_min > 0.10 ORDER BY pct_gap_vs_min DESC"),
    ("Top customers by margin in BC last quarter",
     f"SELECT customer_name, segment, MEASURE(gross_margin) AS gross_margin, MEASURE(net_sales) AS net_sales, MEASURE(gross_margin_pct) AS gross_margin_pct FROM {G}.t_sales_metrics WHERE region = 'BC' AND order_quarter = add_months(date_trunc('QUARTER', current_date()), -3) GROUP BY ALL ORDER BY gross_margin DESC LIMIT 10"),
    ("Monthly net sales and gross margin % for the last 12 months",
     f"SELECT order_month, MEASURE(net_sales) AS net_sales, MEASURE(gross_margin_pct) AS gross_margin_pct FROM {G}.t_sales_metrics WHERE order_date >= add_months(date_trunc('MONTH', current_date()), -12) GROUP BY ALL ORDER BY order_month"),
    ("Backorder rate by warehouse and category this quarter",
     f"SELECT warehouse, category, MEASURE(backorder_rate) AS backorder_rate, MEASURE(units_backordered) AS units_backordered FROM {G}.t_sales_metrics WHERE order_quarter = date_trunc('QUARTER', current_date()) GROUP BY ALL ORDER BY backorder_rate DESC"),
    ("Which SKUs are below reorder point in Toronto?",
     f"SELECT sku, product_name, category, available_qty, on_order_qty, reorder_point, days_of_cover, stock_status FROM {G}.gold_inventory_position WHERE warehouse = 'Toronto' AND below_reorder_point ORDER BY days_of_cover NULLS FIRST"),
    ("Gross margin % by category over the last 12 months",
     f"SELECT category, MEASURE(net_sales) AS net_sales, MEASURE(gross_margin_pct) AS gross_margin_pct FROM {G}.t_sales_metrics WHERE order_date > add_months(current_date(), -12) GROUP BY ALL ORDER BY gross_margin_pct"),
    ("What are the top service ticket themes this quarter?",
     f"SELECT theme, count(*) AS tickets, count_if(is_open) AS open_tickets, round(avg(days_to_close), 1) AS avg_days_to_close FROM {G}.gold_ticket_insights WHERE created_date >= date_trunc('QUARTER', current_date()) GROUP BY theme ORDER BY tickets DESC"),
    ("A customer sent a photo of an electronic keypad entry lever lock in satin nickel - what's our closest equivalent and how does our price compare? (closest equivalent of a described product)",
     equivalent_sql("Electronic Keypad Lock", "Satin Stainless")),
]
sales_samples = ["Which SKUs are priced more than 10% above competitors?", "Top customers by margin in BC last quarter",
                 "A customer sent a photo of a satin nickel passage knob - what's our closest equivalent and how does our price compare?",
                 "Why are Toronto door closer backorders so high?", "What are the top service ticket themes this quarter?"]
sales_benchmarks = [
    ("How many exit device SKUs are more than 10% above the cheapest competitor?", f"SELECT count(*) AS skus FROM {G}.gold_price_benchmark WHERE category = 'Exit Devices' AND pct_gap_vs_min > 0.10"),
    ("What was net revenue last year?", f"SELECT MEASURE(net_sales) AS net_sales FROM {G}.t_sales_metrics WHERE order_year = year(current_date()) - 1"),
    ("Which warehouse has the higher backorder rate?", f"SELECT warehouse, MEASURE(backorder_rate) AS backorder_rate FROM {G}.t_sales_metrics GROUP BY ALL ORDER BY backorder_rate DESC LIMIT 1"),
    ("What's our GM% on faucets?", f"SELECT MEASURE(gross_margin_pct) AS gross_margin_pct FROM {G}.t_sales_metrics WHERE category = 'Faucets'"),
    ("Who is the cheapest competitor most often?", f"SELECT cheapest_competitor, count(*) AS skus FROM {G}.gold_price_benchmark GROUP BY ALL ORDER BY skus DESC LIMIT 1"),
    ("A customer's photo shows an electronic keypad entry lever lock with a 10-button keypad and key override, satin chrome / satin nickel finish, residential style. What is our closest equivalent and how does its price compare to competitors?",
     equivalent_sql("Electronic Keypad Lock", "Satin Stainless")),
    ("Photo from a customer: a round passage door knob on a round rose, polished chrome, no lock. What do we sell that's equivalent, and how is it priced vs competitors?",
     equivalent_sql("Passage Lever Set", "Polished Chrome")),
]
sales_tables = [f"{G}.t_sales_metrics", f"{G}.gold_price_benchmark", f"{G}.gold_price_benchmark_by_competitor", f"{G}.gold_product_margin",
                f"{G}.gold_inventory_position", f"{G}.gold_ticket_insights", f"{G}.gold_dim_product", f"{G}.gold_dim_customer"]
sales_id = upsert_space("T Sales & Ops",
    "TAI backend for sales, margin, competitor pricing, inventory and service questions at T Industries (door hardware and bathroom fixtures, BC / ON / US). Built on governed gold tables and the t_sales_metrics metric view.",
    space_json(sales_tables, sales_instructions, sales_examples, sales_samples, [], sales_benchmarks))

In [0]:
# Knowledge Assistant replacement: the agent reads the raw SharePoint files in the `knowledge` volume (Agent mode)
# and combines them with governed ticket / product tables. The parsed-document tables (silver_doc_chunks,
# gold_doc_summaries) are deliberately NOT attached, so document answers come only from the raw files.
KNOWLEDGE = f"/Volumes/{C}/t_hardware/knowledge"
docs_instructions = [
    "You answer product, installation, warranty, returns, pricing-policy and HR questions for T Industries, a Canadian distributor of door hardware and bathroom fixtures, and combine them with service-ticket and product data.",
    f"Documents: the attached volume {KNOWLEDGE} holds the SharePoint library as raw files - product spec sheets (PDF), installation SOPs (DOCX), WP-01 Warranty Policy (PDF), CS-04 Returns and RMA SOP (DOCX), Pricing and Discount Authority Matrix (DOCX), the Canadian employee handbook excerpt (PDF) and business-rule definitions (Markdown). Answer policy, spec, SOP and HR questions from these files and always cite the document name (and page or section when available). WP-01 is the authoritative warranty source; spec sheets summarise it. If the documents do not contain the answer, say so - do not guess.",
    f"Data: {G}.gold_ticket_insights has one row per HubSpot service ticket with an AI-classified theme (Finish defect, Backorder / late delivery, Wrong item shipped, Damaged in transit, Installation question, Warranty claim, Pricing / invoice dispute), product category, SKU, region and status. {G}.gold_dim_product is the product catalog (category, subcategory, finish, ANSI grade, list price). Use them for counts, trends and which products are affected; join on sku.",
    "For questions that need both, answer the document part from the files and the numbers from the tables, and present them together.",
    "Product vocabulary: 'closer' = door closer; 'panic bar' / 'crash bar' = exit device; 'commercial' = ANSI/BHMA Grade 1; 'residential' = Grade 3. 'This quarter' = the current calendar quarter.",
    "Show percentages with one decimal and money in CAD.",
]
docs_examples = [
    ("How many warranty-claim tickets did we get on door closers in the last 12 months?",
     f"SELECT count(*) AS warranty_tickets, count_if(is_open) AS open_tickets, round(avg(days_to_close), 1) AS avg_days_to_close FROM {G}.gold_ticket_insights WHERE theme = 'Warranty claim' AND category = 'Door Closers' AND created_date > add_months(current_date(), -12)"),
    ("Which product categories had the most warranty claims in the last 12 months?",
     f"SELECT category, count(*) AS warranty_tickets, count_if(is_open) AS open_tickets FROM {G}.gold_ticket_insights WHERE theme = 'Warranty claim' AND created_date > add_months(current_date(), -12) GROUP BY category ORDER BY warranty_tickets DESC"),
    ("Which door closer SKUs have the most installation-question tickets?",
     f"SELECT t.sku, p.product_name, p.subcategory, count(*) AS installation_tickets FROM {G}.gold_ticket_insights t JOIN {G}.gold_dim_product p ON p.sku = t.sku WHERE t.theme = 'Installation question' AND p.category = 'Door Closers' GROUP BY ALL ORDER BY installation_tickets DESC LIMIT 10"),
]
docs_samples = ["What is the restocking fee for returns?",
                "Summarize the DC-Series commercial door closer spec sheet",
                "What's the warranty on commercial door closers, and how many warranty-claim tickets did we get on closers in the last 12 months?",
                "How many vacation days do employees in BC get after one year?",
                "Who can approve a 35% discount off list?"]
docs_id = upsert_space("T Docs & SOPs",
    "TAI backend for product, installation, warranty, returns, pricing-policy and HR questions. Reads the raw SharePoint documents in the knowledge volume directly and combines them with HubSpot ticket and product data.",
    space_json([f"{G}.gold_ticket_insights", f"{G}.gold_dim_product"], docs_instructions, docs_examples, docs_samples,
               volumes=[(KNOWLEDGE, "SharePoint document library (raw PDF, DOCX and Markdown files): spec sheets, installation SOPs, warranty and returns policies, pricing authority matrix, HR handbook excerpt, business rules")]))

In [0]:
DASH = dbutils.widgets.get("dashboard_name")
dash = [d for d in w.lakeview.list() if d.display_name and d.display_name.endswith(DASH)]
for d in dash:
    full = w.lakeview.get(d.dashboard_id)
    spec = json.loads(full.serialized_dashboard)
    if spec.get("uiSettings", {}).get("genieSpace", {}).get("overrideId") == sales_id:
        # Already linked via the bundle's .lvdash.json - don't touch the bundle-managed dashboard (avoids deploy drift)
        print("dashboard already linked:", d.display_name, "->", sales_id); continue
    # New workspace / new space id: link it, and put this id into src/dashboards/t_commercial_overview.lvdash.json
    spec.setdefault("uiSettings", {})["genieSpace"] = {"isEnabled": True, "overrideId": sales_id, "enablementMode": "ENABLED"}
    api("PATCH", f"/api/2.0/lakeview/dashboards/{d.dashboard_id}", body={"serialized_dashboard": json.dumps(spec)})
    try: w.lakeview.publish(d.dashboard_id, warehouse_id=WH, embed_credentials=True)
    except Exception as e: print("publish skipped:", e)
    print("linked dashboard", d.display_name, "->", sales_id, "- update uiSettings.genieSpace.overrideId in the .lvdash.json to match")
if not dash: print(f"dashboard '{DASH}' not found - deploy the bundle first")
dbutils.jobs.taskValues.set("sales_space_id", sales_id); dbutils.jobs.taskValues.set("docs_space_id", docs_id)
print(json.dumps({"T Sales & Ops": sales_id, "T Docs & SOPs": docs_id}, indent=1))